### Keras Tuner
#### Decide number of hidden layers and neruons in neural network

In [1]:
!pip install keras-tuner

In [2]:
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from keras_tuner.tuners import RandomSearch

In [3]:
df = pd.read_csv('/content/Real_Combine.csv')
df.head()

,T,TM,Tm,SLP,H,VV,V,VM,PM 2.5
0,7.4,9.8,4.8,1017.6,93.0,0.5,4.3,9.4,219.720833
1,7.8,12.7,4.4,1018.5,87.0,0.6,4.4,11.1,182.187500
2,6.7,13.4,2.4,1019.4,82.0,0.6,4.8,11.1,154.037500
3,8.6,15.5,3.3,1018.7,72.0,0.8,8.1,20.6,223.208333
4,12.4,20.9,4.4,1017.3,61.0,1.3,8.7,22.2,200.645833


In [4]:
df[df.isnull().any(axis= 1)]

,T,TM,Tm,SLP,H,VV,V,VM,PM 2.5
184,14.3,19.2,10.9,1020.5,91.0,1.6,4.8,11.1,NaN


In [5]:
# df.iloc[[184]]
df.fillna(0, inplace=True)
df[df.isnull().any(axis=1)]

,T,TM,Tm,SLP,H,VV,V,VM,PM 2.5


In [6]:
# dependent and independent features
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

### Hyperparameters
- How many number of hidden layers we should have?
- How many number of neurons we should have in hidden layers?
- Learning Rate

In [7]:
def build_model(hp):
  model = keras.Sequential()
  for i in range(hp.Int('num_layers', 2, 20)):
    model.add(layers.Dense(units= hp.Int('units_' + str(i), min_value = 32, max_value = 512, step = 32), activation= 'relu'))
    model.add(layers.Dense(1, activation= 'linear'))
    model.compile(
        optimizer = keras.optimizers.Adam(hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])),
        loss = 'mean_absolute_error',
        metrics = ['mean_absolute_error']
    )
    return model

In [8]:
tuner = RandomSearch(
    build_model,
    objective = 'val_mean_absolute_error',
    max_trials = 5,
    executions_per_trial = 3,
    directory = 'project',
    project_name = 'Air Quality Index')

Reloading Tuner from project/Air Quality Index/tuner0.json


In [9]:
tuner.search_space_summary()

Search space summary
Default search space size: 3
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 2, 'max_value': 20, 'step': 1, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}


In [10]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.3, random_state= 0)

In [11]:
tuner.search(X_train, y_train, epochs = 5, validation_data = (X_test, y_test))

In [12]:
tuner.results_summary()

Results summary
Results in project/Air Quality Index
Showing 10 best trials
Objective(name="val_mean_absolute_error", direction="min")

Trial 1 summary
Hyperparameters:
num_layers: 3
units_0: 352
learning_rate: 0.001
Score: 59.55276234944662

Trial 0 summary
Hyperparameters:
num_layers: 3
units_0: 320
learning_rate: 0.001
Score: 60.17350514729818

Trial 3 summary
Hyperparameters:
num_layers: 10
units_0: 512
learning_rate: 0.0001
Score: 63.54809824625651

Trial 4 summary
Hyperparameters:
num_layers: 16
units_0: 128
learning_rate: 0.0001
Score: 63.98718770345052

Trial 2 summary
Hyperparameters:
num_layers: 18
units_0: 416
learning_rate: 0.0001
Score: 64.27049128214519
